In [2]:
# Core
import pandas as pd
import numpy as np
from dotenv import load_dotenv

# Pretty printing indented tables
from textwrap import indent

# Database
from sqlalchemy import create_engine, text
import os

'''************************************* DB CONNECTION ***************************************************

#Goal:
Establish a secure connection to the PostgreSQL database using credentials stored 
in an environment variable (DB_URL). 

Storing credentials as environment variables prevents exposing sensitive information 
like usernames, passwords, and host details directly in the source code, ensuring 
better security and easier configuration across different environments (local, staging, production).

#*******************************************************************************************************'''


In [3]:
'''************************************* DB CONNECTION *********************************'''

# 1) Connect to DB  (edit credentials as needed)

#### Database Connection Setup
load_dotenv()
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

DB_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(DB_URL, pool_pre_ping=True)

print("The connection to the DB has been successfully established ✅")


The connection to the DB has been successfully established ✅


In [4]:
'''************************************* DATA LOADING *********************************'''

print("data loading started.......")
# columns you asked for (quoted where needed, aliased to clean names)
COLS_SQL = """
    hospital_name,
    "ZIP4"                           AS zip4,
    setting,
    standard_charge_gross,
    standard_charge_discounted_cash,
    payer_name,
    plan_name,
    standard_charge_negotiated_dollar,
    standard_charge_negotiated_percentage,
    standard_charge_min,
    standard_charge_max,
    bucket,
    specification,
    cpt_code,
    "Simple Description"                       AS category,
    "Specialty"                      AS specialty,
    "Rate_using_min"                 AS rate_using_min,
    "Rate_using_max"                 AS rate_using_max
"""

query = text(f"SELECT {COLS_SQL} FROM standardized_cpt_columns")

# stream from Postgres in chunks, then concatenate (reduce chunk size if needed)
chunks = pd.read_sql(query, engine, chunksize=100_000)
df = pd.concat(chunks, ignore_index=True)

print("data loading successful ✅ ")

data loading started.......


C:\Users\gio12\AppData\Local\Temp\ipykernel_9576\3071433832.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(chunks, ignore_index=True)


data loading successful ✅ 


In [5]:
#*********************************************coerce numerical columns*******************************

print("numeric conversion started ✅ ")
numeric_cols = [
    "standard_charge_gross",
    "standard_charge_discounted_cash",
    "standard_charge_negotiated_dollar",
    "estimated_amount",
    "standard_charge_min",
    "standard_charge_max",
    "Rate_using_min",
    "Rate_using_max"
    
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("numeric conversion successful ✅ ")


numeric conversion started ✅ 
numeric conversion successful ✅ 


==================== PART A : BASIC SUMMARY STATISTICS =========================================================================================================
This section prints:
1) Number of rows & columns
2) Null value count per column (and % missing)
3) Data types overview (and counts by dtype family)
4) Unique counts for key identifiers:
   - hospital_name, payer_name, plan_name, cpt_code, specialty
================================================================================================================================================

In [6]:


print("\n" + "="*80)
print("A) SHAPE OF DATA (rows, columns)")
print("="*80)
n_rows, n_cols = df.shape
print(f"→ The dataset has {n_rows:,} rows and {n_cols:,} columns.\n")

print("="*80)
print("B) MISSING VALUES OVERVIEW (by column)")
print("="*80)
# Build a summary table with null counts and percentages, sorted by highest % missing
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
null_summary = (
    pd.DataFrame({"null_count": null_counts, "null_pct": null_pct})
    .sort_values("null_pct", ascending=False)
)
print("→ Null count and % of missing values for each column (sorted by % missing):")
print(null_summary.to_string())
print()  # spacing

print("="*80)
print("C) DATA TYPES (per column) + dtype family counts")
print("="*80)
# Print the dtype of each column
dtype_series = df.dtypes.astype(str)
print("→ Data types for each column:")
print(dtype_series.to_string())
print()

# Also provide a quick tally of major dtype families (object, float, int, bool, datetime, etc.)
dtype_family_counts = dtype_series.value_counts()
print("→ Count of columns by dtype family:")
print(dtype_family_counts.to_string())
print()

# Optional: quick view of numeric vs. categorical column counts
num_cols = df.select_dtypes(include=["number"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
dt_cols  = df.select_dtypes(include=["datetime64[ns]", "datetimetz"]).columns.tolist()
print(f"→ Numeric columns: {len(num_cols)}")
print(f"→ Categorical (object/category) columns: {len(cat_cols)}")
print(f"→ Datetime columns: {len(dt_cols)}\n")

print("="*80)
print("D) UNIQUE COUNTS FOR KEY IDENTIFIERS")
print("="*80)

def safe_unique_count(frame: pd.DataFrame, col_name: str) -> int | None:
    """
    Returns the number of unique non-null values for `col_name` if the column exists.
    Prints a friendly note if it doesn't, and returns None.
    """
    if col_name in frame.columns:
        return frame[col_name].nunique(dropna=True)
    else:
        print(f"• Column '{col_name}' was NOT found in the dataframe.")
        return None

key_cols = ["hospital_name", "payer_name", "plan_name", "cpt_code", "specialty"]

unique_counts = {}
for c in key_cols:
    unique_counts[c] = safe_unique_count(df, c)

# Pretty-print the results
print("→ Unique non-null value counts:")
for c in key_cols:
    val = unique_counts[c]
    if val is not None:
        print(f"• {c}: {val:,}")

# (Optional) Show the top 5 most frequent values for each key column for quick intuition
print("\n→ Top 5 most frequent values (for columns that exist):")
for c in key_cols:
    if c in df.columns:
        vc = df[c].value_counts(dropna=True).head(5)
        print(f"\n• {c} (top 5):")
        print(indent(vc.to_string(), prefix="  "))



# --- Additional Datapoint: Missing negotiated rates  ---
print("="*80)
print("E) MISSING NEGOTIATED RATES ")
print("="*80)

col_dollar = "standard_charge_negotiated_dollar"
col_pct = "standard_charge_negotiated_percentage"

if col_dollar in df.columns and col_pct in df.columns:
    # Rows where BOTH negotiated dollar and percentage are null
    mask_missing_negotiated = df[col_dollar].isna() & df[col_pct].isna()
    missing_count = mask_missing_negotiated.sum()
    total_rows = len(df)
    missing_pct = (missing_count / total_rows * 100).round(2)

    print(f"→ Rows with BOTH negotiated dollar and percentage missing: {missing_count:,}")
    print(f"→ That’s {missing_pct}% of all records with no negotiated price information.")
else:
    print("⚠️ Columns for negotiated rates not found — skipping this datapoint.")
    
print("\n✅ Basic summary statistics complete.\n")



A) SHAPE OF DATA (rows, columns)
→ The dataset has 4,754,000 rows and 18 columns.

B) MISSING VALUES OVERVIEW (by column)
→ Null count and % of missing values for each column (sorted by % missing):
                                       null_count  null_pct
standard_charge_negotiated_percentage     3700907     77.85
standard_charge_negotiated_dollar         3585543     75.42
standard_charge_discounted_cash           2150695     45.24
standard_charge_gross                     2113562     44.46
standard_charge_min                        249848      5.26
standard_charge_max                        213408      4.49
rate_using_min                              21438      0.45
rate_using_max                              21438      0.45
zip4                                         9423      0.20
plan_name                                     542      0.01
payer_name                                     11      0.00
bucket                                         11      0.00
specification        

 #========================= PART B: MEANINGFUL NUMERIC INSIGHTS =========================
 Goal: Move beyond .describe() and compute interpretable pricing insights:
   1) Pricing variability (std, coefficient of variation, ranges, IQR)
   2) Extremes: top/bottom negotiated prices with context (hospital/payer/CPT)
   3) (Optional but useful) Quick view by payer bucket (Commercial/Government/Self-pay)
#======================================================================================

In [7]:

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# ---- 0) Choose price columns that exist in your DF (robust to missing columns) -------
candidate_charge_cols = [
    "standard_charge_gross",
    "standard_charge_discounted_cash",
    "standard_charge_negotiated_dollar",
    "standard_charge_min",
    "standard_charge_max",
    "Rate_using_min",
    "Rate_using_max",
]
charge_cols = [c for c in candidate_charge_cols if c in df.columns]

print("\n" + "="*100)
print("PART B — MEANINGFUL NUMERIC INSIGHTS")
print("="*100)
print(f"Using {len(charge_cols)} charge columns:", charge_cols)

# ---- 1) Overall focused summary (mean/median/std/min/max + CV + Range + IQR) ---------
print("\n" + "-"*100)
print("B1) OVERALL SUMMARY with CV (std/mean), Range, and IQR (75th–25th)")
print("-"*100)

summary = df[charge_cols].agg(["count", "mean", "median", "std", "min", "max"])
# Add CV, Range, IQR
q25 = df[charge_cols].quantile(0.25)
q75 = df[charge_cols].quantile(0.75)
cv = summary.loc["std"] / summary.loc["mean"]
_rng = summary.loc["max"] - summary.loc["min"]
iqr = q75 - q25

# Build a tidy table
summary_extended = (
    pd.concat(
        {
            "count": summary.loc["count"],
            "mean": summary.loc["mean"],
            "median": summary.loc["median"],
            "std": summary.loc["std"],
            "cv_std_over_mean": cv,
            "min": summary.loc["min"],
            "q25": q25,
            "q75": q75,
            "iqr": iqr,
            "max": summary.loc["max"],
            "range_max_minus_min": _rng,
        },
        axis=1,
    )
    .round(2)
)

print("→ Focused numeric snapshot (columns as rows):")
print(indent(summary_extended.to_string(), "  "))


g = "standard_charge_gross"
cash = "standard_charge_discounted_cash"
neg = "standard_charge_negotiated_dollar"


PART B — MEANINGFUL NUMERIC INSIGHTS
Using 5 charge columns: ['standard_charge_gross', 'standard_charge_discounted_cash', 'standard_charge_negotiated_dollar', 'standard_charge_min', 'standard_charge_max']

----------------------------------------------------------------------------------------------------
B1) OVERALL SUMMARY with CV (std/mean), Range, and IQR (75th–25th)
----------------------------------------------------------------------------------------------------
→ Focused numeric snapshot (columns as rows):
                                           count     mean   median       std  cv_std_over_mean  min    q25      q75      iqr          max  range_max_minus_min
  standard_charge_gross             2,640,438.00 2,112.31   415.00  6,321.16              2.99 1.00 274.00 1,053.00   779.00   168,116.14           168,115.14
  standard_charge_discounted_cash   2,603,305.00 1,220.53   264.74  3,027.48              2.48 0.20 178.10   756.76   578.66    57,686.02            57,685.82
 

#========================= PART B+: SPECIALTY & CPT FORMATTED SUMMARIES  =========================
#Uses your exact specialty spellings and normalizes df['specialty'] to these canonical labels.

In [8]:


SPECIALTIES_CANON = [
    "Dermatology",
    "Ophthalmology",
    "Orthopedic",
    "Physical Medicine & Rehabilitation",
    "Psychiatry/Psychology",
    "Radiation Oncology",
]

# --- normalize specialty values in df to the canonical set above ---
import re

def _norm_key(s: str) -> str:
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    s = str(s).strip().lower()
    s = s.replace("&", "and")
    s = s.replace("/", " / ")
    s = re.sub(r"\s+", " ", s)  # collapse spaces
    return s

# build a lookup: normalized -> canonical label
canon_lookup = {
    _norm_key("Dermatology"): "Dermatology",
    _norm_key("Ophthalmology"): "Ophthalmology",
    _norm_key("Orthopedic"): "Orthopedic",
    _norm_key("Physical Medicine & Rehabilitation"): "Physical Medicine & Rehabilitation",
    _norm_key("Psychiatry/Psychology"): "Psychiatry/Psychology",
    _norm_key("Radiation Oncology"): "Radiation Oncology",
}

# create a canonical specialty column in df (non-destructive)
if "specialty" in df.columns:
    df["specialty_canon"] = df["specialty"].map(lambda x: canon_lookup.get(_norm_key(x), None))
else:
    df["specialty_canon"] = None  # keeps functions from breaking gracefully

# ---- helper to build the same stats table you used in Part B ----
def _make_numeric_summary(frame: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    cols = [c for c in cols if c in frame.columns]
    if not cols:
        return pd.DataFrame()

    summary = frame[cols].agg(["count", "mean", "median", "std", "min", "max"])
    q25 = frame[cols].quantile(0.25)
    q75 = frame[cols].quantile(0.75)
    cv  = summary.loc["std"] / summary.loc["mean"]
    rng = summary.loc["max"] - summary.loc["min"]
    iqr = q75 - q25

    out = (
        pd.concat(
            {
                "count": summary.loc["count"],
                "mean": summary.loc["mean"],
                "median": summary.loc["median"],
                "std": summary.loc["std"],
                "cv_std_over_mean": cv,
                "min": summary.loc["min"],
                "q25": q25,
                "q75": q75,
                "iqr": iqr,
                "max": summary.loc["max"],
                "range_max_minus_min": rng,
            },
            axis=1,
        )
        .round(2)
    )
    return out

def specialty_and_cpt_report(df: pd.DataFrame, canonical_specialty: str, charge_cols: list[str],
                             cpt_list: list[str] | None = None, top_k_cpts: int = 2):
    # filter to the canonical specialty
    df_s = df[df.get("specialty_canon").eq(canonical_specialty)].copy()
    if df_s.empty:
        print(f"\n⚠️ No rows for specialty '{canonical_specialty}'.")
        return

    # ---- Specialty summary (Gio format)
    print("\n" + canonical_specialty.upper() + " SUMMARY")
    spec_tbl = _make_numeric_summary(df_s, charge_cols)
    print(indent(spec_tbl.to_string() if not spec_tbl.empty else "  (no numeric columns found)", "  "))

    # ---- CPT summaries
    if "cpt_code" not in df_s.columns:
        print("\n⚠️ No 'cpt_code' column — skipping CPT summaries.")
        return

    if not cpt_list:
        vc = df_s["cpt_code"].value_counts(dropna=True)
        cpt_list = vc.head(max(top_k_cpts, 1)).index.tolist()

    print("\n### CPTCODE SUMMARY\n")
    print(canonical_specialty.upper())

    for cpt in cpt_list:
        df_c = df_s[df_s["cpt_code"] == cpt]
        if df_c.empty:
            continue
        print(f"\nCPT_CODE: {cpt}")
        cpt_tbl = _make_numeric_summary(df_c, charge_cols)
        print(indent(cpt_tbl.to_string(), "  "))

# ----------------- RUN REPORTS FOR ALL YOUR CANONICAL SPECIALTIES -----------------
# Reuses your existing `charge_cols` from Part B
for spec in SPECIALTIES_CANON:
    specialty_and_cpt_report(df, canonical_specialty=spec, charge_cols=charge_cols, top_k_cpts=2)



DERMATOLOGY SUMMARY
                                         count     mean   median       std  cv_std_over_mean   min      q25       q75       iqr        max  range_max_minus_min
  standard_charge_gross             196,899.00 1,840.81   837.00  4,201.35              2.28 25.00   586.00  1,091.00    505.00  70,760.84            70,735.84
  standard_charge_discounted_cash   196,899.00   979.03   544.05  1,831.13              1.87  5.00   250.00    676.42    426.42  17,874.25            17,869.25
  standard_charge_negotiated_dollar 149,521.00 2,531.17   579.60  6,709.97              2.65  0.95   242.73  2,324.00  2,081.27 938,390.00           938,389.05
  standard_charge_min               804,184.00 1,814.06 1,728.00  1,455.13              0.80  0.95   676.42  2,702.00  2,025.58  28,304.34            28,303.39
  standard_charge_max               804,184.00 9,661.26 7,108.00 17,208.93              1.78  4.98 1,234.05 11,371.00 10,136.95 938,390.00           938,385.02

### CPTCODE SUMMAR

In [9]:

# ---- 2) Extremes: top/bottom negotiated records with context -------------------------
print("\n" + "-"*100)
print("B4) EXTREMES: Highest and Lowest negotiated prices with hospital/payer/CPT context")
print("-"*100)

context_cols = [c for c in ["hospital_name", "payer_name", "plan_name", "cpt_code", "bucket", "setting", "specialty"] if c in df.columns]

def show_extremes(col, k=5):
    if col not in df.columns:
        print(f"• Skipped {col}: not in dataframe.")
        return
    # Drop NA to avoid noisy results
    sub = df[context_cols + [col]].dropna(subset=[col])
    if sub.empty:
        print(f"• Skipped {col}: no non-null values.")
        return

    print(f"\n→ TOP {k} by '{col}':")
    topk = sub.nlargest(k, col)
    print(indent(topk.to_string(index=False), "  "))

    print(f"\n→ BOTTOM {k} by '{col}':")
    botk = sub.nsmallest(k, col)
    print(indent(botk.to_string(index=False), "  "))

# Show extremes for negotiated dollar first (most decision-relevant)
if neg in df.columns:
    show_extremes(neg, k=5)
# Optionally also for discounted cash and gross
if cash in df.columns:
    show_extremes(cash, k=5)
if g in df.columns:
    show_extremes(g, k=5)

# ---- 4) (Optional) Quick view by payer bucket ----------------------------------------
print("\n" + "-"*100)
print("B5) QUICK BY-BUCKET SUMMARY (Commercial vs Government vs Self-Pay)")
print("-"*100)

if "bucket" in df.columns:
    # Aggregate medians (robust to outliers) + counts, then sort for readability
    agg_map = {}
    if g in df:    agg_map[g] = "median"
    if cash in df: agg_map[cash] = "median"
    if neg in df:  agg_map[neg] = "median"

    if agg_map:
        bucket_summary = (
            df.groupby("bucket")
              .agg({**agg_map, **{"cpt_code": "count"}})
              .rename(columns={"cpt_code": "row_count"})
              .sort_values(by=list(agg_map.keys())[0], ascending=False)
              .round(2)
        )
        print("→ Median charges by bucket (+ row counts):")
        print(indent(bucket_summary.to_string(), "  "))

        # Also show average discount % by bucket (if we computed them)
        disc_cols = [c for c in ["discount_cash_pct", "discount_negotiated_pct"] if c in df.columns]
        if disc_cols:
            bucket_discounts = df.groupby("bucket")[disc_cols].median().round(2)
            print("\n→ Median discount % by bucket:")
            print(indent(bucket_discounts.to_string(), "  "))
    else:
        print("• Skipped: no charge columns available to aggregate.")
else:
    print("• Skipped: 'bucket' column not found.")

print("\n✅ Part B complete — produced interpretable metrics .")



----------------------------------------------------------------------------------------------------
B4) EXTREMES: Highest and Lowest negotiated prices with hospital/payer/CPT context
----------------------------------------------------------------------------------------------------

→ TOP 5 by 'standard_charge_negotiated_dollar':
                hospital_name                          payer_name   plan_name cpt_code     bucket    setting  specialty  standard_charge_negotiated_dollar
  AdventHealth North Pinellas blue_cross_&_blue_shield_of_florida traditional    29827 Commercial outpatient Orthopedic                       9,593,845.00
  AdventHealth North Pinellas blue_cross_&_blue_shield_of_florida traditional    29827 Commercial outpatient Orthopedic                       9,593,845.00
  AdventHealth North Pinellas blue_cross_&_blue_shield_of_florida traditional    29881 Commercial outpatient Orthopedic                       5,449,586.00
  AdventHealth North Pinellas blue_cross_&_bl

 ====================== PART C : DATA SUMMARY ===========================
 Goal:
   - Number of hospitals
   - Number of payers (Commercial / Government / Self-pay)
   - Mean & median prices by CPT codes
   - DISTINCT payer counts per bucket
 ======================================================================

In [10]:

print("\n" + "="*100)
print("PART C — DATA SUMMARY")
print("="*100)

# ---- Column references ----
col_hosp = "hospital_name"
col_bucket = "bucket"
col_payer = "payer_name"
col_cpt = "cpt_code"
col_gross = "standard_charge_gross"
col_cash = "standard_charge_discounted_cash"
col_neg = "standard_charge_negotiated_dollar"
col_final_rate = "standard_charge_max"

# =============================  Number of hospitals =============================
if col_hosp in df.columns:
    n_hospitals = df[col_hosp].nunique(dropna=True)
    print(f"\n→ Total unique hospitals represented: {n_hospitals:,}")
else:
    print("\n⚠️ 'hospital_name' column not found — skipping hospital count.")

# =============================  CPT-level mean & median prices =============================
print("\n" + "-"*100)
print("D3) Mean & Median Prices by CPT Code")
print("-"*100)

price_cols = [col_gross, col_cash, col_neg, col_final_rate]
existing_price_cols = [c for c in price_cols if c in df.columns]

if col_cpt in df.columns and existing_price_cols:
    # Compute mean and median for each CPT code
    cpt_summary = (
        df.groupby(col_cpt)[existing_price_cols]
          .agg(["mean", "median", "count"])
          .round(2)
    )

    # Flatten column names for readability
    cpt_summary.columns = ['_'.join(col).strip() for col in cpt_summary.columns.values]
    cpt_summary = cpt_summary.reset_index()

    print("→ CPT-level summary (showing top 10 rows):")
    print(indent(cpt_summary.head(10).to_string(index=False), "  "))

# ==================== DISTINCT PAYERS BY BUCKET  ====================

from textwrap import indent

COL_BUCKET = "bucket"
COL_PAYER  = "payer_name"

# --- 0) Normalize bucket labels so similar terms collapse properly ---
bucket_map = {
    "commercial": "Commercial",
    "comm": "Commercial",
    "gov": "Government",
    "government": "Government",
    "medicare": "Government",
    "medicaid": "Government",
    "self pay": "Self Pay",
    "self-pay": "Self Pay",
    "selfpay": "Self Pay",
    "cash": "Self Pay",
}
def normalize_bucket(s):
    if pd.isna(s):
        return "Unknown"
    key = str(s).strip().lower()
    return bucket_map.get(key, s if s in ["Commercial", "Government", "Self Pay"] else "Unknown")

df["_bucket_norm"] = df[COL_BUCKET].apply(normalize_bucket)

# --- 1️⃣ DISTINCT payer counts per bucket ---
payers_by_bucket = (
    df.dropna(subset=[COL_PAYER])
      .groupby("_bucket_norm")[COL_PAYER]
      .nunique()
      .reset_index(name="unique_payers")
      .sort_values("unique_payers", ascending=False)
)

print("\n→ DISTINCT payers by bucket:")
print(indent(payers_by_bucket.to_string(index=False), "  "))

# --- 2️⃣ Roster preview: show first 10 payer names per bucket ---
roster = (
    df.dropna(subset=[COL_PAYER])
      .groupby("_bucket_norm")[COL_PAYER]
      .apply(lambda s: sorted(s.dropna().unique().tolist()))
      .reset_index(name="payer_list")
)


print("\n✅ Part C complete.")



PART C — DATA SUMMARY

→ Total unique hospitals represented: 101

----------------------------------------------------------------------------------------------------
D3) Mean & Median Prices by CPT Code
----------------------------------------------------------------------------------------------------
→ CPT-level summary (showing top 10 rows):
  cpt_code  standard_charge_gross_mean  standard_charge_gross_median  standard_charge_gross_count  standard_charge_discounted_cash_mean  standard_charge_discounted_cash_median  standard_charge_discounted_cash_count  standard_charge_negotiated_dollar_mean  standard_charge_negotiated_dollar_median  standard_charge_negotiated_dollar_count  standard_charge_max_mean  standard_charge_max_median  standard_charge_max_count
     10040                    1,244.02                        589.00                         3897                                411.16                                  235.60                                   3897                  

In [11]:
# ========================= PART C+: G SUMMARIES (OVERALL, BY BUCKET, CPT IN BUCKET) =========================

# pick the working frame (use capped if present)
_ddd = df_cap if 'df_cap' in globals() else df

# choose numeric charge columns that exist
_charge_candidates = [
    "standard_charge_gross",
    "standard_charge_discounted_cash",
    "standard_charge_negotiated_dollar",
    "standard_charge_min",
    "standard_charge_max",
    "Rate_using_min",
    "Rate_using_max",
]
_charge_cols = [c for c in _charge_candidates if c in _ddd.columns]

def _make_numeric_summary(frame: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    cols = [c for c in cols if c in frame.columns]
    if not cols or frame.empty:
        return pd.DataFrame()
    summary = frame[cols].agg(["count", "mean", "median", "std", "min", "max"])
    q25 = frame[cols].quantile(0.25)
    q75 = frame[cols].quantile(0.75)
    cv  = summary.loc["std"] / summary.loc["mean"]
    rng = summary.loc["max"] - summary.loc["min"]
    iqr = q75 - q25
    out = (
        pd.concat(
            {
                "count": summary.loc["count"],
                "mean": summary.loc["mean"],
                "median": summary.loc["median"],
                "std": summary.loc["std"],
                "cv_std_over_mean": cv,
                "min": summary.loc["min"],
                "q25": q25,
                "q75": q75,
                "iqr": iqr,
                "max": summary.loc["max"],
                "range_max_minus_min": rng,
            },
            axis=1,
        ).round(2)
    )
    return out

# ---------- 1) OVERALL SUMMARY (Gio table) ----------
print("\nOVERALL SUMMARY")
overall_tbl = _make_numeric_summary(_ddd, _charge_cols)
print(indent(overall_tbl.to_string() if not overall_tbl.empty else "  (no numeric columns found)", "  "))

# ---------- 2) BY BUCKET SUMMARY (uses your normalized column if present) ----------
bucket_col = "_bucket_norm" if "_bucket_norm" in _ddd.columns else "bucket"
if bucket_col in _ddd.columns:
    for b in _ddd[bucket_col].dropna().unique():
        sub = _ddd[_ddd[bucket_col] == b]
        print(f"\n{str(b).upper()} SUMMARY")
        b_tbl = _make_numeric_summary(sub, _charge_cols)
        print(indent(b_tbl.to_string() if not b_tbl.empty else "  (no data for this bucket)", "  "))
else:
    print("\n⚠️ No bucket column present for by-bucket summaries.")

# ---------- 3) CPTCODE SUMMARY BY BUCKET (top 2 CPTs per bucket by volume) ----------
if bucket_col in _ddd.columns and "cpt_code" in _ddd.columns:
    print("\n### CPTCODE SUMMARY BY BUCKET")
    for b in _ddd[bucket_col].dropna().unique():
        sub = _ddd[_ddd[bucket_col] == b]
        # pick top 2 CPTs by count to keep output readable (adjust as needed)
        top_cpts = sub["cpt_code"].value_counts(dropna=True).head(2).index.tolist()
        if not top_cpts:
            continue
        print(f"\n{str(b).upper()}")
        for cpt in top_cpts:
            sub_cpt = sub[sub["cpt_code"] == cpt]
            print(f"\nCPT_CODE: {cpt}")
            cpt_tbl = _make_numeric_summary(sub_cpt, _charge_cols)
            print(indent(cpt_tbl.to_string() if not cpt_tbl.empty else "  (no numeric columns for this CPT)", "  "))
else:
    print("\n⚠️ Missing bucket or cpt_code column — skipping CPT-by-bucket summaries.")



OVERALL SUMMARY
                                           count     mean   median       std  cv_std_over_mean  min    q25      q75      iqr          max  range_max_minus_min
  standard_charge_gross             2,640,438.00 2,112.31   415.00  6,321.16              2.99 1.00 274.00 1,053.00   779.00   168,116.14           168,115.14
  standard_charge_discounted_cash   2,603,305.00 1,220.53   264.74  3,027.48              2.48 0.20 178.10   756.76   578.66    57,686.02            57,685.82
  standard_charge_negotiated_dollar 1,168,457.00 2,259.44   246.60 18,339.86              8.12 0.95 111.24 1,421.00 1,309.76 9,593,845.00         9,593,844.05
  standard_charge_min               4,504,152.00 1,508.28   346.58  2,558.88              1.70 0.01 114.12 1,950.00 1,835.88    65,394.70            65,394.69
  standard_charge_max               4,540,592.00 7,806.33 1,100.70 73,849.19              9.46 0.01 274.00 7,448.00 7,174.00 9,593,845.00         9,593,844.99

COMMERCIAL SUMMARY
         

c:\Users\gio12\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


#========================= PART D: PRICE SUMMARY BY SPECIALTY =========================
Goal: Compute mean, median, and std deviation of all numeric charge columns
across each specialty. Helps identify which specialties have higher or
more variable pricing patterns.
#=====================================================================================

In [12]:

print("\n" + "="*100)
print("PART D — PRICE SUMMARY BY SPECIALTY")
print("="*100)

num_cols = [
    "standard_charge_gross",
    "standard_charge_discounted_cash",
    "standard_charge_negotiated_dollar",
    "standard_charge_min",
    "standard_charge_max",
    "Rate_using_min",
    "Rate_using_max"  # <- you kept this
]
existing_num_cols = [c for c in num_cols if c in df.columns]

# (Optional but safe) force Rate_using_max to numeric if it's stored as text
if "Rate_using_max" in df.columns:
    df["Rate_using_max"] = pd.to_numeric(df["Rate_using_max"], errors="coerce")

if "specialty" in df.columns and existing_num_cols:

    specialty_summary = (
        df.groupby("specialty")[existing_num_cols]
          .agg(["mean", "median", "std"])
          .round(2)
    )

    specialty_summary.columns = [f"{col}_{stat}" for col, stat in specialty_summary.columns]
    specialty_summary = specialty_summary.reset_index()

    print("\n→ Mean, Median, and Standard Deviation of price metrics by specialty:")
    print(indent(specialty_summary.to_string(index=False), "  "))

    # ---- Top specialties by average negotiated price (if present)
    if "standard_charge_negotiated_dollar_mean" in specialty_summary.columns:
        top_specialties = (
            specialty_summary.sort_values("standard_charge_negotiated_dollar_mean", ascending=False)
            [["specialty", "standard_charge_negotiated_dollar_mean"]]
            .head(10)
        )
        print("\n→ Top 10 specialties by average negotiated price:")
        print(indent(top_specialties.to_string(index=False), "  "))

    # ---- NEW: Top specialties by average Rate_using_max
    if "Rate_using_max_mean" in specialty_summary.columns:
        top_rum = (
            specialty_summary.sort_values("Rate_using_max_mean", ascending=False)
            [["specialty", "Rate_using_max_mean"]]
            .head(10)
        )
        print("\n→ Top 10 specialties by average Rate_using_max:")
        print(indent(top_rum.to_string(index=False), "  "))

    # ---- Highest variability (std) in negotiated prices
    if "standard_charge_negotiated_dollar_std" in specialty_summary.columns:
        var_specialties = (
            specialty_summary.sort_values("standard_charge_negotiated_dollar_std", ascending=False)
            [["specialty", "standard_charge_negotiated_dollar_std"]]
            .head(10)
        )
        print("\n→ Specialties with highest variability (std) in negotiated prices:")
        print(indent(var_specialties.to_string(index=False), "  "))

    # ---- NEW: Highest variability (std) in Rate_using_max
    if "Rate_using_max_std" in specialty_summary.columns:
        var_rum = (
            specialty_summary.sort_values("Rate_using_max_std", ascending=False)
            [["specialty", "Rate_using_max_std"]]
            .head(10)
        )
        print("\n→ Specialties with highest variability (std) in Rate_using_max:")
        print(indent(var_rum.to_string(index=False), "  "))

else:
    print("⚠️ Missing 'specialty' column or no numeric charge columns found.")


print("\n✅ Part D complete.")




PART D — PRICE SUMMARY BY SPECIALTY

→ Mean, Median, and Standard Deviation of price metrics by specialty:
                           specialty  standard_charge_gross_mean  standard_charge_gross_median  standard_charge_gross_std  standard_charge_discounted_cash_mean  standard_charge_discounted_cash_median  standard_charge_discounted_cash_std  standard_charge_negotiated_dollar_mean  standard_charge_negotiated_dollar_median  standard_charge_negotiated_dollar_std  standard_charge_min_mean  standard_charge_min_median  standard_charge_min_std  standard_charge_max_mean  standard_charge_max_median  standard_charge_max_std
                         Dermatology                    1,840.81                        837.00                   4,201.35                                979.03                                  544.05                             1,831.13                                2,531.17                                    579.60                               6,709.97                  1

========================= PART F: OUTLIER HANDLING & GROUP-LEVEL SUMMARIES =========================
Goal: 
Clean numeric columns by removing placeholder values and winsorizing (1st–99th percentile)
within each hospital to control extreme outliers, while preserving real hospital-level variation. 
Normalize key categorical fields (bucket, setting, specialty) for consistent grouping. 

Then compute descriptive summaries (mean, median, std, count) across major dimensions:
  • By CPT code  
  • By payer bucket  
  • By specialty  
  • By care setting  
  • By hospital and bucket combination  
  • By CPT code and bucket combination  


====================================================================================


## OVERALL GOOD STATISTIC ON RATE_MIN AND RATE_MAX

In [13]:

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# --- helper: case/space-insensitive column matcher ------------------------------------
def _norm(s: str) -> str:
    return str(s).strip().lower().replace(" ", "_")

def find_present(df, desired_names):
    # desired_names can be strings or lists of aliases
    present = []
    df_map = {_norm(c): c for c in df.columns}  # normalized -> actual
    for d in desired_names:
        aliases = [d] if isinstance(d, str) else d
        # try each alias normalized
        for a in aliases:
            a_norm = _norm(a)
            if a_norm in df_map:
                present.append(df_map[a_norm])
                break  # take the first alias that exists
    return present

# ---------------------------------------------------------------
# 0) Columns setup (robust to missing columns)
# ---------------------------------------------------------------
rate_cols = find_present(df, [["Rate_using_min","rate_using_min"],
                              ["Rate_using_max","rate_using_max"]])

extra_numeric = find_present(
    df,
    [
        ["standard_charge_gross","gross","list_price","standard_gross"],
        ["standard_charge_discounted_cash","cash_price","discounted_cash","self_pay"],
        ["standard_charge_min","min_rate","negotiated_min"],
        ["standard_charge_max","max_rate","negotiated_max"],
    ],
)

num_cols = rate_cols + [c for c in extra_numeric if c not in rate_cols]

cat_needed = find_present(
    df,
    [
        ["hospital_name","hospital","facility_name"],
        ["bucket","payer_bucket","payer_type"],
        ["specialty","service_line","dep_specialty"],
        ["setting","place_of_service","inpatient_outpatient"],
        ["cpt_code","cpt","hcpcs","hcpcs_code","code"],
    ],
)
missing_cats = set(["hospital_name","bucket","specialty","setting","cpt_code"]) - set(
    [_norm(c) for c in cat_needed]
)
if missing_cats:
    print(f"⚠️ Missing categorical columns (canonical names): {sorted(missing_cats)}")

print(f"Using rate columns: {rate_cols}")
print(f"Using numeric columns: {num_cols}")

# ---------------------------------------------------------------
# 1) Light cleaning: remove placeholders so they don't affect quantiles
#    (coerce to numeric first so comparisons work even if strings)
# ---------------------------------------------------------------
PLACEHOLDER_BIG = 999_999_999.0
df_cap = df.copy()

for col in num_cols:
    df_cap[col] = pd.to_numeric(df_cap[col], errors="coerce")
    df_cap.loc[df_cap[col] <= 0, col] = np.nan
    df_cap.loc[df_cap[col] == PLACEHOLDER_BIG, col] = np.nan

# ---------------------------------------------------------------
# 2) Groupwise winsorization (1st–99th pct) WITHIN EACH HOSPITAL
# ---------------------------------------------------------------
def winsorize_series(s, p_low=0.01, p_high=0.99):
    s = pd.to_numeric(s, errors="coerce")
    if s.notna().sum() < 5:  # too few points; skip
        return s
    ql, qh = s.quantile([p_low, p_high])
    return s.clip(lower=ql, upper=qh)

impact_rows = []
# find actual hospital_name column used above
hospital_col = find_present(df_cap, [["hospital_name","hospital","facility_name"]])
hospital_col = hospital_col[0] if hospital_col else None

if hospital_col:
    for col in num_cols:
        before = df_cap[col].copy()
        df_cap[col] = (
            df_cap.groupby(hospital_col, group_keys=False)[col]
                  .apply(winsorize_series)
        )
        changed = (before.notna() & df_cap[col].notna() & (before != df_cap[col])).sum()
        impact_rows.append([col, int(before.notna().sum()), int(changed),
                            float(before.quantile(0.01)), float(before.quantile(0.99))])
else:
    # Fallback: global winsorization if hospital name missing
    for col in num_cols:
        before = df_cap[col].copy()
        df_cap[col] = winsorize_series(df_cap[col])
        changed = (before.notna() & df_cap[col].notna() & (before != df_cap[col])).sum()
        impact_rows.append([col, int(before.notna().sum()), int(changed),
                            float(before.quantile(0.01)), float(before.quantile(0.99))])

impact = pd.DataFrame(
    impact_rows, columns=["column", "n_non_null", "n_capped", "global_q01", "global_q99"]
).set_index("column")
print("\n" + "-"*100)
print("Outlier treatment (winsorization within hospital):")
print(indent(impact.round(2).to_string(), "  "))

# ---------------------------------------------------------------
# 3) Helper to summarize groups
# ---------------------------------------------------------------
def group_summary(df_in, group_cols, value_cols):
    # resolve group columns using the same finder
    resolved_groups = find_present(df_in, group_cols)
    if len(resolved_groups) != len(group_cols):
        missing = [gc for gc in group_cols if gc not in resolved_groups]
        return f"⚠️ Skipped {group_cols} — missing after resolution: {missing}"
    vals_present = [c for c in value_cols if c in df_in.columns]
    if not vals_present:
        return f"⚠️ No value columns present among {value_cols}"
    out = (
        df_in.groupby(resolved_groups)[vals_present]
             .agg(["mean", "median", "std", "count"])
             .round(2)
    )
    return out
####################################################################
# ---------------------------------------------------------------
# 3.5) Normalize category values so 'both', 'Both', 'BOTH' group together
# ---------------------------------------------------------------
def norm_cat(x: str) -> str:
    if pd.isna(x): 
        return np.nan
    s = str(x).strip().lower().replace("-", " ").replace("_", " ")
    s = " ".join(s.split())  # collapse multiple spaces
    return s

# Canonical maps
setting_map = {
    "inpatient": "inpatient",
    "outpatient": "outpatient",
    "both": "both",
}

bucket_map = {
    "commercial": "commercial",
    "government": "government",
    "other government": "government",  # optional: fold into government if you prefer
    "self pay": "self pay",
    "self-pay": "self pay",
    "selfpay": "self pay",
    "unknown": "unknown",
}

# Specialty canonicalization (use the six you listed)
specialty_map = {
    "dermatology": "dermatology",
    "ophthalmology": "ophthalmology",
    "orthopedics": "orthopedics",
    "physical medicine and rehabilitation": "physical medicine and rehabilitation",
    "psychiatry/psychology": "psychiatry/psychology",
    "radiation oncology": "radiation oncology",
}

# Create normalized columns (leave originals untouched)
if "setting" in df_cap.columns:
    df_cap["setting_norm"] = df_cap["setting"].map(lambda v: setting_map.get(norm_cat(v), norm_cat(v)))

if "bucket" in df_cap.columns:
    df_cap["bucket_norm"] = df_cap["bucket"].map(lambda v: bucket_map.get(norm_cat(v), norm_cat(v)))

if "specialty" in df_cap.columns:
    df_cap["specialty_norm"] = df_cap["specialty"].map(lambda v: specialty_map.get(norm_cat(v), norm_cat(v)))

# ---------------------------------------------------------------
# 4) Produce the tables you asked for
# ---------------------------------------------------------------
tables = {}
tables["by_cpt_code"]        = group_summary(df_cap, ["cpt_code"], rate_cols)
tables["by_bucket"]          = group_summary(df_cap, ["bucket_norm"], rate_cols)
tables["by_specialty"]       = group_summary(df_cap, ["specialty_norm"], rate_cols)
tables["by_setting"]         = group_summary(df_cap, ["setting_norm"], rate_cols)
tables["by_hospital_bucket"] = group_summary(df_cap, ["hospital_name", "bucket_norm"], rate_cols)
tables["by_cpt_bucket"]      = group_summary(df_cap, ["cpt_code", "bucket_norm"], rate_cols)

# ---------------------------------------------------------------
# 5) Display nicely (no files written)
# ---------------------------------------------------------------
for name, tbl in tables.items():
    print("\n" + "="*100)
    print(name.replace("_", " ").upper())
    print("="*100)
    if isinstance(tbl, str):
        print(tbl)  # message about missing columns
    else:
        print(indent(tbl.head(20).to_string(), "  "))



Using rate columns: ['rate_using_min', 'rate_using_max']
Using numeric columns: ['rate_using_min', 'rate_using_max', 'standard_charge_gross', 'standard_charge_discounted_cash', 'standard_charge_min', 'standard_charge_max']

----------------------------------------------------------------------------------------------------
Outlier treatment (winsorization within hospital):
                                   n_non_null  n_capped  global_q01  global_q99
  column                                                                       
  rate_using_min                      4732562     73668       14.22   18,385.00
  rate_using_max                      4732562     56044       18.90   42,325.00
  standard_charge_gross               2640438      5128       94.50   31,776.00
  standard_charge_discounted_cash     2603305      5117       49.32   16,323.59
  standard_charge_min                 4504152     37841       10.63   10,866.00
  standard_charge_max                 4540592     28258       24